# 02 -- Stage 2: DFDC Frame Extraction & Training (Kaggle)

Designed to run as a **Kaggle Notebook** with a DFDC dataset attached via
"Add Input" (no download/upload needed -- Kaggle mounts it read-only
under `/kaggle/input/...`).

**Handles both DFDC packaging styles**:
- Pre-extracted: `.mp4`/`.avi`/`.mov` files + `metadata.json` directly under the input folder.
- **ZIP archives** (e.g. the common "first N parts" reuploads, ~10GB/part):
  each zip is extracted one at a time into scratch space, processed, then
  **deleted** before moving to the next -- this keeps peak disk usage to
  roughly one part's size, regardless of how many parts you attach (attaching
  all ~100GB at once would otherwise exceed Kaggle's working-disk quota).

Labels come from Kaggle's real `metadata.json` format:
`{"filename.mp4": {"label": "REAL"|"FAKE", ...}}`.

**Run order note**: if you're running this *before* Stage 1 (no
`models/image_model.pth` yet), that's fine -- this notebook falls back to
randomly-initialized weights and trains standalone. Stage 3's merge step
reconciles image + video training regardless of which you ran first.

**If the full 10-part set is taking many hours**: extraction alone can take
~6 hours, and CPU training can take ~2.8 hours *per epoch* -- together
that exceeds a single Kaggle session. If a single run of this notebook
keeps getting cut off before training finishes, stop running extraction
and training together: finish extraction-only as its own successful
commit (accelerator set to CPU), turn its output into a Kaggle Dataset,
then use `02b_stage2_train_from_extracted_frames.ipynb` (GPU-enabled, no
re-extraction) for the training half.

In [ ]:
import sys, os

IS_COLAB = os.path.exists("/var/colab/hostname")
IS_KAGGLE = (not IS_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab" if IS_KAGGLE else "/content/ForensicAI_Lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/MRoy03/Forensic_LAB.git {PROJECT_ROOT}

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
elif IS_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Not in Colab/Kaggle -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import torch, cv2, numpy as np, pandas as pd, json, shutil, zipfile
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

from config.loader import get_settings, resolve_dataset_root, resolve_path
from services.video.extractor import extract_frames, get_video_fps, get_frame_count
from services.video.sampler import sample_frame_indices
from services.preprocessing.pipeline import preprocess_image
from services.detection.model import ForensicMultiHeadNet

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)

# Raw DFDC frames are full video stills (e.g. 1920x1080) where the face --
# the only region face-swap artifacts actually live in -- occupies a small
# fraction of the frame. Resizing the WHOLE frame straight to a 256x256
# training input (as earlier versions of this notebook did) squashes that
# region down to a blurry handful of pixels, which is the likely reason
# balanced_acc plateaued near chance across several sampler/loss tuning
# rounds: the signal to learn from was never really there. Cropping to the
# detected face (with margin) before resizing keeps it at near-full
# resolution instead. Haar cascade (bundled with opencv, zero extra
# dependency/download) is good enough for DFDC's mostly-frontal talking-head
# footage; falls back to the full frame if no face is found so a bad
# detection never silently drops a sample.
FACE_CASCADE = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
FACE_CROP_MARGIN = 0.4  # extra context around the detected box, as a fraction of its size


def crop_to_face(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = FACE_CASCADE.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    if len(faces) == 0:
        return frame, False
    x, y, w, h = max(faces, key=lambda b: b[2] * b[3])  # largest detected face
    mx, my = int(w * FACE_CROP_MARGIN), int(h * FACE_CROP_MARGIN)
    x0, y0 = max(0, x - mx), max(0, y - my)
    x1, y1 = min(frame.shape[1], x + w + mx), min(frame.shape[0], y + h + my)
    return frame[y0:y1, x0:x1], True

**If you already have frames extracted from an earlier run of this
notebook (before face-cropping was added), they need to be
re-extracted** -- the old frames were saved already-shrunk to 256x256
with no crop, so the face-region detail needed for this fix is gone
from those files; there's no way to recover it after the fact.

Before re-running the full ~6-hour extraction, validate the fix cheaply
first: set `MAX_METADATA_FILES = 1`, `MAX_VIDEOS_PER_PART = 50` below,
extract that small batch, and retrain Stage 2 on just it for a couple
epochs. Watch the printed `face detected in X/Y frames` rate (should be
high, e.g. >80%, for DFDC's mostly-frontal footage -- a low rate means
the cascade isn't finding faces and the crop is silently falling back to
full frames) and whether `balanced_acc` clears the ~55-59% ceiling seen
across every prior sampler/loss-only attempt. Only commit to the full
re-extraction once that small run looks meaningfully better.

## Locate the attached dataset

Change `KAGGLE_INPUT_SLUG` to match whatever "Add Input" shows in your
notebook's sidebar. Falls back to `dataset_root/dfdc_subset` if not
running on Kaggle (e.g. you manually placed files there instead).

In [ ]:
import os


def find_files(root: Path, suffix: str) -> list[Path]:
    """Recursively find files by suffix via os.walk(followlinks=True)
    rather than pathlib glob. Kaggle mounts attached datasets via symlinks
    (sometimes through multiple nested mount layers), and pathlib's
    recursive glob -- even with recurse_symlinks=True -- has proven
    unreliable at following these in practice; os.walk's followlinks is a
    much simpler, longer-established mechanism that just works."""
    matches = []
    for dirpath, _dirnames, filenames in os.walk(root, followlinks=True):
        for f in filenames:
            if f.lower().endswith(suffix):
                matches.append(Path(dirpath) / f)
    return matches


KAGGLE_INPUT_SLUG = "dfdc-10"  # <-- change to your attached dataset's folder name
kaggle_input_path = Path("/kaggle/input") / KAGGLE_INPUT_SLUG

zip_source_dir = kaggle_input_path if kaggle_input_path.exists() else dataset_root / "dfdc_subset"
print("Reading DFDC data from:", zip_source_dir)

zip_files = sorted(find_files(zip_source_dir, ".zip"))
if zip_files:
    print(f"Found {len(zip_files)} zip part(s):")
    for z in zip_files:
        print(f"  {z.name}  ({z.stat().st_size / 1e9:.1f} GB)")
else:
    print("No .zip files found -- assuming the dataset is already extracted at this path.")
    print("If that's wrong, here's everything actually found under", zip_source_dir, "(first 20):")
    _all_files = find_files(zip_source_dir, "")
    for f in _all_files[:20]:
        print("  ", f)
    if not _all_files:
        print("  (nothing at all -- check KAGGLE_INPUT_SLUG matches `!ls /kaggle/input/` exactly)")

## Process one metadata.json at a time, saving after each

Each `metadata.json` found (whether freshly extracted from a zip, or
already sitting there because Kaggle auto-extracted this dataset on
attach) is its own independently-saved unit -- progress is written to
disk immediately after each one finishes, not just at the very end. If
this cell is interrupted or the session restarts, re-running it skips
whatever's already done instead of losing everything or starting over.

`MAX_METADATA_FILES` and `MAX_VIDEOS_PER_PART` let you do a cheap first
test run before committing to the full dataset -- the defaults below are
deliberately small; raise both once you've confirmed it works end-to-end.

In [ ]:
MAX_METADATA_FILES = 2   # how many metadata.json "parts" to process this run; None = all
MAX_VIDEOS_PER_PART = 50  # videos per metadata.json this run; None = all
FRAMES_PER_VIDEO = 5

scratch_dir = Path("/kaggle/working/dfdc_scratch")
frames_out_dir = resolve_path("data/synthetic_data")
frames_out_dir.mkdir(parents=True, exist_ok=True)
frame_manifest_path = resolve_path("data/dfdc_frame_manifest.csv")
progress_path = resolve_path("data/dfdc_processed_parts.txt")

processed_parts = set(progress_path.read_text().splitlines()) if progress_path.exists() else set()
total_rows_written = sum(1 for _ in open(frame_manifest_path)) - 1 if frame_manifest_path.exists() else 0
if processed_parts:
    print(f"Resuming: {total_rows_written} rows already written, {len(processed_parts)} part(s) already done.")


def process_metadata_file(meta_path, part_id, max_videos):
    video_folder = meta_path.parent
    with open(meta_path) as f:
        metadata = json.load(f)
    items = list(metadata.items())
    if max_videos is not None:
        items = items[:max_videos]

    rows = []
    faces_found = faces_total = 0
    for filename, info in items:
        video_path = video_folder / filename
        if not video_path.exists():
            continue
        label = "real" if info["label"].upper() == "REAL" else "fake"
        manip_type = "none" if label == "real" else "gan"  # deepfake/face-swap -> closest existing class

        fps = get_video_fps(str(video_path))
        total = get_frame_count(str(video_path))
        indices = set(sample_frame_indices(total, fps, settings.video.sample_fps, max_frames=FRAMES_PER_VIDEO))
        for idx, frame in extract_frames(str(video_path)):
            if idx not in indices:
                continue
            frame_name = f"{part_id}_{video_path.stem}_frame{idx}.jpg"
            frame, found_face = crop_to_face(frame)
            faces_total += 1
            faces_found += found_face
            # Resize to the model's actual input size before saving --
            # preprocess_image() resizes to this anyway at training time,
            # so saving at full video resolution only wastes disk with no
            # quality benefit (this cut a ~17.5GB extraction down sharply
            # in practice, and sped up every subsequent read too).
            frame = cv2.resize(frame, tuple(settings.inference.image_size))
            cv2.imwrite(str(frames_out_dir / frame_name), frame)
            rows.append({"filename": str(frames_out_dir / frame_name), "label": label, "type": manip_type})
    if faces_total:
        print(f"  face detected in {faces_found}/{faces_total} frames ({faces_found / faces_total:.0%})")
    return rows


def save_part(part_id, rows):
    global total_rows_written
    pd.DataFrame(rows, columns=["filename", "label", "type"]).to_csv(
        frame_manifest_path, mode="a", header=(total_rows_written == 0), index=False
    )
    total_rows_written += len(rows)
    with open(progress_path, "a") as f:
        f.write(part_id + "\n")
    print(f"  {len(rows)} rows (running total: {total_rows_written})")


processed_count = 0

if zip_files:
    # Zip-packaged: extract one zip at a time into scratch, process its
    # metadata.json(s), delete scratch, then move to the next zip.
    for zip_path in zip_files:
        part_id = zip_path.stem
        if part_id in processed_parts:
            print(f"[SKIP] {part_id} already done")
            continue
        if MAX_METADATA_FILES is not None and processed_count >= MAX_METADATA_FILES:
            print(f"Reached MAX_METADATA_FILES={MAX_METADATA_FILES} for this run, stopping here.")
            break

        print(f"\n=== {part_id} (from zip) ===")
        if scratch_dir.exists():
            shutil.rmtree(scratch_dir)
        scratch_dir.mkdir(parents=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(scratch_dir)

        rows = []
        for meta_path in find_files(scratch_dir, "metadata.json"):
            rows.extend(process_metadata_file(meta_path, part_id, MAX_VIDEOS_PER_PART))
        save_part(part_id, rows)
        shutil.rmtree(scratch_dir)
        processed_count += 1
else:
    # Already extracted (e.g. Kaggle auto-extracted this on attach): find
    # every metadata.json directly and save progress after each one.
    all_metadata_files = find_files(zip_source_dir, "metadata.json")
    print(f"Found {len(all_metadata_files)} metadata.json file(s) under {zip_source_dir}")

    for meta_path in all_metadata_files:
        part_id = meta_path.parent.name  # e.g. "dfdc_train_part_0"
        if part_id in processed_parts:
            print(f"[SKIP] {part_id} already done")
            continue
        if MAX_METADATA_FILES is not None and processed_count >= MAX_METADATA_FILES:
            print(f"Reached MAX_METADATA_FILES={MAX_METADATA_FILES} for this run, stopping here.")
            break

        print(f"\n=== {part_id} ===")
        rows = process_metadata_file(meta_path, part_id, MAX_VIDEOS_PER_PART)
        save_part(part_id, rows)
        processed_count += 1

print(f"\nDone (for this run). {total_rows_written} total frame rows written to {frame_manifest_path}")

## Train on the extracted frames

In [ ]:
class FrameDataset(Dataset):
    """Like Stage 1's image dataset, but reads full paths straight from the
    frame manifest (no images_dir join) and has no ground-truth masks --
    mask loss is zero-weighted for every sample via has_mask=0."""

    def __init__(self, manifest_csv, image_size):
        self.df = pd.read_csv(manifest_csv)
        self.image_size = image_size
        self.label_to_idx = {l: i for i, l in enumerate(settings.labels.binary)}
        manip_classes = settings.labels.manipulation_types
        self.manip_to_idx = {m: i for i, m in enumerate(manip_classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_bgr = cv2.imread(row["filename"])
        if image_bgr is None:
            print(f"[WARN] Could not read {row['filename']}, substituting another sample")
            return self[(idx + 1) % len(self)]
        x = preprocess_image(image_bgr, tuple(self.image_size))

        label_idx = self.label_to_idx[row["label"]]
        row_type = row["type"]
        # Membership test (not == none_type) also catches other
        # non-manipulation-class values, e.g. the feedback pipeline's
        # "unknown" fallback for a fake sample with no specified type.
        has_manip = 1.0 if row_type in self.manip_to_idx else 0.0
        manip_idx = self.manip_to_idx.get(row_type, 0)
        mask = np.zeros((1, *self.image_size), dtype=np.float32)

        return (
            torch.from_numpy(x),
            torch.tensor(label_idx, dtype=torch.long),
            torch.tensor(manip_idx, dtype=torch.long),
            torch.from_numpy(mask),
            torch.tensor(0.0, dtype=torch.float32),
            torch.tensor(has_manip, dtype=torch.float32),
        )

In [ ]:
frame_dataset = FrameDataset(frame_manifest_path, settings.inference.image_size)
loader = DataLoader(frame_dataset, batch_size=16, shuffle=True, num_workers=2)
print("Frame dataset size:", len(frame_dataset))

In [ ]:
# pretrained=True loads ImageNet weights into the encoder (timm adapts
# the first conv layer to in_chans=5 automatically). Every run before
# this was trained from random init -- none of these notebooks actually
# passed `pretrained=` through, so it silently used the class default
# (False) regardless of this setting. A from-scratch CNN needs far more
# data than these datasets provide; this is likely the dominant reason
# results stayed near chance even after fixing sampling/loss weighting.
model = ForensicMultiHeadNet(
    backbone=settings.models.backbone,
    in_chans=settings.inference.input_channels,
    num_manip_classes=len(settings.labels.manipulation_types),
    pretrained=settings.models.pretrained_encoder,
).to(device)

image_ckpt = resolve_path(settings.models.image_model_checkpoint)
if image_ckpt.exists():
    model.load_state_dict(torch.load(image_ckpt, map_location=device))
    print("Warm-started from Stage 1 checkpoint:", image_ckpt)
else:
    print("No Stage 1 checkpoint found -- training from randomly-initialized weights. "
          "That's fine; Stage 3's merge step reconciles image + video training either order.")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
cls_loss_fn = torch.nn.CrossEntropyLoss()
manip_loss_fn = torch.nn.CrossEntropyLoss(reduction="none")
mask_loss_fn = torch.nn.BCEWithLogitsLoss(reduction="none")

NUM_EPOCHS = 5  # adjust based on dataset size / Kaggle session limits

In [ ]:
model.train()
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    for x, label_idx, manip_idx, mask, has_mask, has_manip in loader:
        x, label_idx, manip_idx = x.to(device), label_idx.to(device), manip_idx.to(device)
        mask, has_mask, has_manip = mask.to(device), has_mask.to(device), has_manip.to(device)

        optimizer.zero_grad()
        out = model(x)

        loss_cls = cls_loss_fn(out["cls_logits"], label_idx)
        manip_loss_per_sample = manip_loss_fn(out["manip_logits"], manip_idx)
        loss_manip = (manip_loss_per_sample * has_manip).sum() / (has_manip.sum() + 1e-8)
        mask_loss_per_px = mask_loss_fn(out["mask_logits"], mask)
        loss_mask = (mask_loss_per_px.mean(dim=[1, 2, 3]) * has_mask).sum() / (has_mask.sum() + 1e-8)

        loss = loss_cls + loss_manip + loss_mask
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} -- loss: {running_loss / len(loader):.4f}")

In [ ]:
checkpoint_path = resolve_path(settings.models.video_model_checkpoint)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), checkpoint_path)
print("Saved checkpoint to", checkpoint_path)

## Getting the checkpoint out of Kaggle

Kaggle keeps this session's output files under the notebook's "Output"
tab, but that's separate from your Google Drive. To use this checkpoint in
the Colab notebooks (Stage 3 merge, Stage 6 export), download
`models/video_model.pth` from the Output tab and upload it to
`ForensicAI_Lab/models/video_model.pth` on your Drive.